# FDOx SPARQL Explorer

This notebook demonstrates how to:
1. Fetch a **FAIR Digital Object (FDOx)** ZIP from [Zenodo](https://zenodo.org) via the REST API
2. Extract and parse the `fdo-metadata.ttl` from the ZIP root
3. Run example **SPARQL queries** on the RDF graph using `rdflib`

Two example FDOx records are used:
- `18369126` — Software FDO (*o3d-epidoc-extractor*)
- `18744133` — 3D Model FDO

**Dependencies:** `rdflib`, `requests`, `pandas`, `tabulate`

In [ ]:
# ── Install dependencies if needed ──────────────────────────────────────────
%pip install rdflib requests pandas tabulate --quiet

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import io
import zipfile
import requests
import pandas as pd
from rdflib import Graph, Namespace, URIRef
from rdflib.plugins.sparql import prepareQuery
from IPython.display import display, Markdown

# ── Namespace declarations (used across FDOx TTL files) ──────────────────────
DCAT   = Namespace("http://www.w3.org/ns/dcat#")
DCT    = Namespace("http://purl.org/dc/terms/")
PROV   = Namespace("http://www.w3.org/ns/prov#")
SCHEMA = Namespace("http://schema.org/")
RDF    = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
RDFS   = Namespace("http://www.w3.org/2000/01/rdf-schema#")
OWL    = Namespace("http://www.w3.org/2002/07/owl#")
XSD    = Namespace("http://www.w3.org/2001/XMLSchema#")
FOAF   = Namespace("http://xmlns.com/foaf/0.1/")
SPDX   = Namespace("http://spdx.org/rdf/terms#")
CODEMETA = Namespace("https://codemeta.github.io/terms/")

print("✅ Imports OK")

## 1 — Helper Functions

In [ ]:
ZENODO_API = "https://zenodo.org/api/records/{record_id}"
TTL_NAME   = "fdo-metadata.ttl"


# ── ZIP Central Directory parser (no full download needed) ───────────────────
# A ZIP file stores its file table (Central Directory) at the *end* of the
# archive. We fetch only the last ~64 KB via an HTTP Range request to find
# the End-of-Central-Directory record, parse offsets, then fetch only the
# bytes belonging to fdo-metadata.ttl.

import struct

EOCD_SIGNATURE  = b'PK\x05\x06'
CD_SIGNATURE    = b'PK\x01\x02'
LOCAL_SIGNATURE = b'PK\x03\x04'


def _get_zip_size(url: str) -> int:
    """HEAD request to get total ZIP size in bytes."""
    r = requests.head(url, timeout=15, allow_redirects=True)
    r.raise_for_status()
    return int(r.headers['Content-Length'])


def _fetch_range(url: str, start: int, end: int) -> bytes:
    """Fetch a byte range from a URL (inclusive end)."""
    r = requests.get(url, headers={"Range": f"bytes={start}-{end}"}, timeout=30)
    if r.status_code not in (200, 206):
        raise IOError(f"Range request failed: HTTP {r.status_code}")
    return r.content


def _find_eocd(tail: bytes) -> dict:
    """Locate and parse the End-of-Central-Directory record in the tail bytes."""
    idx = tail.rfind(EOCD_SIGNATURE)
    if idx == -1:
        raise ValueError("EOCD signature not found — not a valid ZIP tail")
    eocd = tail[idx:]
    # EOCD layout: sig(4) disk(2) start_disk(2) entries_disk(2) entries_total(2)
    #              cd_size(4) cd_offset(4) comment_len(2)
    _, _, _, _, entries, cd_size, cd_offset, _ = struct.unpack_from('<4sHHHHIIH', eocd)
    return {"entries": entries, "cd_size": cd_size, "cd_offset": cd_offset}


def _parse_central_directory(cd_bytes: bytes) -> list[dict]:
    """Parse all Central Directory file headers into a list of dicts."""
    entries = []
    pos = 0
    while pos < len(cd_bytes):
        if cd_bytes[pos:pos+4] != CD_SIGNATURE:
            break
        # CD header: sig(4) ver_made(2) ver_need(2) flags(2) compress(2)
        #            mod_time(2) mod_date(2) crc(4) comp_size(4) uncomp_size(4)
        #            fname_len(2) extra_len(2) comment_len(2) disk_start(2)
        #            int_attr(2) ext_attr(4) local_hdr_offset(4)
        hdr = struct.unpack_from('<4s6H3I5HII', cd_bytes, pos)
        fname_len    = hdr[10]
        extra_len    = hdr[11]
        comment_len  = hdr[12]
        comp_size    = hdr[8]
        uncomp_size  = hdr[9]
        compress     = hdr[5]
        local_offset = hdr[16]
        hdr_size = 46
        fname = cd_bytes[pos+hdr_size : pos+hdr_size+fname_len].decode('utf-8', errors='replace')
        entries.append({
            "name":         fname,
            "compress":     compress,
            "comp_size":    comp_size,
            "uncomp_size":  uncomp_size,
            "local_offset": local_offset,
        })
        pos += hdr_size + fname_len + extra_len + comment_len
    return entries


def _extract_entry_via_range(url: str, entry: dict) -> bytes:
    """Fetch and decompress a single ZIP entry using a Range request."""
    import zlib
    # Local file header: sig(4) ver(2) flags(2) compress(2) time(2) date(2)
    #                    crc(4) comp_size(4) uncomp_size(4) fname_len(2) extra_len(2)
    local_hdr_raw = _fetch_range(url, entry['local_offset'], entry['local_offset'] + 29)
    if local_hdr_raw[:4] != LOCAL_SIGNATURE:
        raise ValueError("Local file header signature mismatch")
    fname_len = struct.unpack_from('<H', local_hdr_raw, 26)[0]
    extra_len = struct.unpack_from('<H', local_hdr_raw, 28)[0]
    data_start = entry['local_offset'] + 30 + fname_len + extra_len
    data_end   = data_start + entry['comp_size'] - 1
    compressed = _fetch_range(url, data_start, data_end)
    if entry['compress'] == 0:       # stored
        return compressed
    elif entry['compress'] == 8:     # deflate
        return zlib.decompress(compressed, -15)
    else:
        raise ValueError(f"Unsupported compression method: {entry['compress']}")


def fetch_ttl_from_zenodo(record_id: str, ttl_name: str = TTL_NAME) -> tuple[str, dict]:
    """Fetch only fdo-metadata.ttl from a Zenodo FDOx record.

    Strategy:
      1. Zenodo Files API  → locate the ZIP download URL + total size
      2. HTTP Range GET    → fetch last 64 KB (contains ZIP Central Directory)
      3. Parse CD          → find byte offset + size of fdo-metadata.ttl
      4. HTTP Range GET    → fetch only that entry's compressed bytes
      5. Decompress        → return TTL string

    Falls back to full ZIP download if the server does not support Range requests.
    """
    # Step 1 — record metadata + ZIP URL
    api_url = ZENODO_API.format(record_id=record_id)
    print(f"📡 Fetching record metadata: {api_url}")
    meta = requests.get(api_url, timeout=30)
    meta.raise_for_status()
    record = meta.json()
    files = record.get("files", [])
    zip_entry = next((f for f in files if f["key"].endswith(".zip")), None)
    if zip_entry is None:
        raise FileNotFoundError(f"No ZIP found in record {record_id}")
    zip_url  = zip_entry["links"]["self"]
    zip_size = zip_entry["size"]
    print(f"📦 ZIP: {zip_entry['key']} ({zip_size / 1e6:.1f} MB)")

    # Step 2 — check Range support
    head = requests.head(zip_url, timeout=15, allow_redirects=True)
    range_ok = head.headers.get("Accept-Ranges", "").lower() == "bytes"

    if range_ok:
        try:
            # Step 2b — fetch tail (last 64 KB covers EOCD + CD for most FDOs)
            tail_size  = min(65536, zip_size)
            tail_start = zip_size - tail_size
            print(f"🔍 Fetching ZIP tail ({tail_size / 1024:.0f} KB via Range)...")
            tail = _fetch_range(zip_url, tail_start, zip_size - 1)

            # Step 3 — parse Central Directory
            eocd   = _find_eocd(tail)
            cd_abs = eocd["cd_offset"]
            print(f"📋 Central Directory: {eocd['entries']} entries at offset {cd_abs}")
            # If CD is within the tail we already have it; otherwise fetch it
            cd_in_tail_start = cd_abs - tail_start
            if cd_in_tail_start >= 0:
                cd_bytes = tail[cd_in_tail_start : cd_in_tail_start + eocd["cd_size"]]
            else:
                print(f"   CD not in tail, fetching separately...")
                cd_bytes = _fetch_range(zip_url, cd_abs, cd_abs + eocd["cd_size"] - 1)
            entries = _parse_central_directory(cd_bytes)

            # Step 4 — find TTL entry
            ttl_entry = next(
                (e for e in entries if e["name"].endswith(ttl_name)
                 and e["name"].count("/") <= 1),
                None
            )
            if ttl_entry is None:
                ttl_entry = next((e for e in entries if e["name"].endswith(ttl_name)), None)
            if ttl_entry is None:
                available = [e['name'] for e in entries]
                raise FileNotFoundError(
                    f"{ttl_name} not found. ZIP contains:\n" + "\n".join(available[:30])
                )
            print(f"✅ Found: {ttl_entry['name']} "
                  f"(compressed: {ttl_entry['comp_size']/1024:.1f} KB, "
                  f"uncompressed: {ttl_entry['uncomp_size']/1024:.1f} KB)")

            # Step 5 — fetch + decompress only that entry
            raw = _extract_entry_via_range(zip_url, ttl_entry)
            print(f"🎯 Downloaded {len(raw)/1024:.1f} KB instead of {zip_size/1e6:.1f} MB")
            return raw.decode("utf-8"), record

        except Exception as e:
            print(f"⚠️  Range strategy failed ({e}), falling back to full download...")

    # Fallback — full ZIP download
    print(f"⬇️  Downloading full ZIP ({zip_size / 1e6:.1f} MB)...")
    r = requests.get(zip_url, timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        candidates = [n for n in zf.namelist() if n.endswith(ttl_name)]
        if not candidates:
            raise FileNotFoundError(f"{ttl_name} not found in ZIP")
        target = min(candidates, key=lambda n: n.count("/"))
        print(f"📄 Extracted: {target}")
        return zf.read(target).decode("utf-8"), record


# ── Well-known namespace URIs for common archaeology / LOD prefixes ─────────
KNOWN_PREFIXES = {
    # CIDOC-CRM family
    "crm":      "http://www.cidoc-crm.org/cidoc-crm/",
    "crmdig":   "http://www.ics.forth.gr/isl/CRMdig/",
    "crmsci":   "http://www.ics.forth.gr/isl/CRMsci/",
    "crmarchaeo": "http://www.cidoc-crm.org/crmarchaeo/",
    "lrmoo":    "http://iflastandards.info/ns/lrm/lrmoo/",
    "frbroo":   "http://iflastandards.info/ns/fr/frbr/frbroo/",
    "frbr":     "http://purl.org/vocab/frbr/core#",
    # Geo / spatial
    "geo":      "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "geosparql": "http://www.opengis.net/ont/geosparql#",
    "sf":       "http://www.opengis.net/ont/sf#",
    "gn":       "http://www.geonames.org/ontology#",
    # Research / Science
    "datacite": "http://purl.org/spar/datacite/",
    "fabio":    "http://purl.org/spar/fabio/",
    "bibo":     "http://purl.org/ontology/bibo/",
    "codemeta": "https://codemeta.github.io/terms/",
    "schema":   "http://schema.org/",
    # Identity / provenance
    "wd":       "http://www.wikidata.org/entity/",
    "wdt":      "http://www.wikidata.org/prop/direct/",
    "gnd":      "https://d-nb.info/gnd/",
    "viaf":     "http://viaf.org/viaf/",
    # Common
    "skos":     "http://www.w3.org/2004/02/skos/core#",
    "vcard":    "http://www.w3.org/2006/vcard/ns#",
    "time":     "http://www.w3.org/2006/time#",
    "org":      "http://www.w3.org/ns/org#",
    "void":     "http://rdfs.org/ns/void#",
    "adms":     "http://www.w3.org/ns/adms#",
    "spdx":     "http://spdx.org/rdf/terms#",
}


def _patch_prefixes(ttl_str: str) -> str:
    """Inject @prefix declarations for any known namespace not already declared."""
    import re
    declared = set(re.findall(r'@prefix\s+(\w+):', ttl_str))
    used      = set(re.findall(r'(?:^|\s)(\w+):[A-Za-z_]', ttl_str))
    missing   = used - declared
    injected  = []
    for pfx in missing:
        if pfx in KNOWN_PREFIXES:
            injected.append(f'@prefix {pfx}: <{KNOWN_PREFIXES[pfx]}> .')
    if injected:
        print(f"🔧 Injecting {len(injected)} missing prefix(es): {', '.join(p.split()[1] for p in injected)}")
        return "\n".join(injected) + "\n" + ttl_str
    return ttl_str


def load_graph(ttl_str: str) -> Graph:
    """Parse a Turtle string into an rdflib Graph.
    
    Automatically injects missing but known @prefix declarations
    before parsing (handles crmdig:, crm:, frbroo: etc.).
    Falls back to line-by-line parsing if the full graph still fails.
    """
    g = Graph()
    patched = _patch_prefixes(ttl_str)
    try:
        g.parse(data=patched, format="turtle")
        print(f"📊 Graph loaded — {len(g)} triples")
        return g
    except Exception as e:
        print(f"⚠️  Full parse failed ({e.__class__.__name__}), attempting tolerant parse...")
    # Tolerant fallback: collect @prefix lines + try each statement block
    prefix_lines = []
    blocks: list[str] = []
    current: list[str] = []
    for line in patched.splitlines():
        stripped = line.strip()
        if stripped.startswith("@prefix") or stripped.startswith("PREFIX"):
            prefix_lines.append(line)
        elif stripped.endswith(".") and not stripped.startswith("#"):
            current.append(line)
            blocks.append("\n".join(current))
            current = []
        elif stripped:
            current.append(line)
    prefix_header = "\n".join(prefix_lines) + "\n"
    ok, skipped = 0, 0
    for block in blocks:
        try:
            g.parse(data=prefix_header + block, format="turtle")
            ok += 1
        except Exception:
            skipped += 1
    print(f"📊 Tolerant parse: {len(g)} triples loaded ({ok} blocks OK, {skipped} skipped)")
    return g


def sparql_table(graph: Graph, query: str, label: str = "", show: bool = True) -> pd.DataFrame:
    """Run a SPARQL SELECT query and return results as a DataFrame.
    Set show=False to suppress display (e.g. when only exporting to CSV).
    """
    if label and show:
        display(Markdown(f"### {label}"))
    results = graph.query(query)
    rows = []
    for row in results:
        rows.append({str(var): str(val) if val is not None else "" for var, val in zip(results.vars, row)})
    df = pd.DataFrame(rows, columns=[str(v) for v in results.vars])
    if show:
        display(df)
    return df


print("✅ Helper functions defined")


---
## 2 — Load FDOs from Zenodo

Choose which record(s) to work with. Both are loaded here; run the cells independently as needed.

In [ ]:
# ── Record IDs ───────────────────────────────────────────────────────────────
RECORDS = {
    "software": "18369126",   # o3d-epidoc-extractor
    "3d_model": "18744133",   # 3D Model FDO
}

In [ ]:
# ── Load Software FDO ────────────────────────────────────────────────────────
print("=" * 60)
print("FDO: SOFTWARE")
print("=" * 60)
ttl_sw, record_sw = fetch_ttl_from_zenodo(RECORDS["software"])
g_sw = load_graph(ttl_sw)

In [ ]:
# ── Load 3D Model FDO ────────────────────────────────────────────────────────
print("=" * 60)
print("FDO: 3D MODEL")
print("=" * 60)
ttl_3d, record_3d = fetch_ttl_from_zenodo(RECORDS["3d_model"])
g_3d = load_graph(ttl_3d)

---
## 3 — Inspect the Raw TTL

Quick look at the Turtle source before querying.

In [ ]:
# ── Show first N lines of the TTL ────────────────────────────────────────────
N_LINES = 60

print("── Software TTL (first lines) ──")
print("\n".join(ttl_sw.splitlines()[:N_LINES]))
print("...")

In [ ]:
print("── 3D Model TTL (first lines) ──")
print("\n".join(ttl_3d.splitlines()[:N_LINES]))
print("...")

---
## 4 — SPARQL Queries

All queries run on both graphs so you can compare the two FDO types side-by-side.

### 4.1 Basic Metadata

In [ ]:
# NOTE: FDOx uses https://schema.org/ (with s), not http://schema.org/
Q_BASIC = """
PREFIX dct:    <http://purl.org/dc/terms/>
PREFIX dcat:   <http://www.w3.org/ns/dcat#>
PREFIX schema: <https://schema.org/>
PREFIX fdo:    <https://w3id.org/fdo-squirrel/>
PREFIX codemeta: <https://codemeta.github.io/terms/>

SELECT ?property ?value
WHERE {
  ?fdo a dcat:Dataset .
  VALUES ?prop {
    dct:title
    dct:description
    dct:created
    dct:modified
    dct:issued
    dct:hasVersion
    dct:type
    dct:license
    fdo:context
    schema:funding
    codemeta:programmingLanguage
    codemeta:codeRepository
  }
  ?fdo ?prop ?val .
  BIND(REPLACE(STR(?prop), "^.*/", "") AS ?property)
  BIND(STR(?val) AS ?value)
}
ORDER BY ?property
"""

display(Markdown("## 4.1 Basic Metadata"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_BASIC)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_BASIC)
_ = None  # suppress Out[] cell output

### 4.2 Creators and Contributors

In [ ]:
# Creators: FDOx stores names as schema:name (https://schema.org/),
# and ORCID URIs are used directly as subject URIs (not via owl:sameAs)
Q_CREATORS = """
PREFIX dct:    <http://purl.org/dc/terms/>
PREFIX dcat:   <http://www.w3.org/ns/dcat#>
PREFIX schema: <https://schema.org/>

SELECT ?name ?orcid ?type
WHERE {
  ?fdo a dcat:Dataset .
  ?fdo dct:creator ?agent .
  OPTIONAL { ?agent schema:name ?name }
  OPTIONAL { ?agent a ?type . FILTER(?type != <http://www.w3.org/2002/07/owl#NamedIndividual>) }
  BIND(STR(?agent) AS ?orcid)
}
ORDER BY ?name
"""

display(Markdown("## 4.2 Creators"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_CREATORS)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_CREATORS)
_ = None  # suppress Out[] cell output

### 4.3 FDO Roles and Types

In [ ]:
# FDO Roles: in FDOx, roles are on distributions via fdo:role (not rdf:type)
# Values observed: "software", "script", "documentation"
Q_ROLES = """
PREFIX dcat:  <http://www.w3.org/ns/dcat#>
PREFIX fdo:   <https://w3id.org/fdo-squirrel/>

SELECT ?role (COUNT(?dist) AS ?count) (SUM(?bytes) AS ?totalBytes)
WHERE {
  ?fdo a dcat:Dataset .
  ?fdo dcat:distribution ?dist .
  ?dist fdo:role ?role .
  OPTIONAL { ?dist dcat:byteSize ?bytes }
}
GROUP BY ?role
ORDER BY DESC(?count)
"""

display(Markdown("## 4.3 FDO Roles (fdo:role) — count & total size"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_ROLES)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_ROLES)
_ = None  # suppress Out[] cell output

### 4.4 Distributions / Files

In [ ]:
# Distributions: FDOx uses fdo:sha256 (not spdx:checksum) and fdo:path
Q_DIST = """
PREFIX dcat:  <http://www.w3.org/ns/dcat#>
PREFIX dct:   <http://purl.org/dc/terms/>
PREFIX fdo:   <https://w3id.org/fdo-squirrel/>

SELECT ?path ?mediaType ?byteSize ?role ?sha256
WHERE {
  ?fdo a dcat:Dataset .
  ?fdo dcat:distribution ?dist .
  OPTIONAL { ?dist fdo:path      ?path }
  OPTIONAL { ?dist dcat:mediaType ?mediaType }
  OPTIONAL { ?dist dcat:byteSize  ?byteSize }
  OPTIONAL { ?dist fdo:role       ?role }
  OPTIONAL { ?dist fdo:sha256     ?sha256 }
}
ORDER BY ?role ?path
LIMIT 30
"""

display(Markdown("## 4.4 Distributions (first 30, ordered by role + path)"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_DIST)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_DIST)
_ = None  # suppress Out[] cell output

### 4.5 Provenance (PROV-O)

In [ ]:
# Spatial & Temporal coverage — FDOx uses geosparql + dct:temporal
# (no prov: vocabulary is used in fdo-metadata.ttl)
Q_SPATIOTEMPORAL = """
PREFIX dct:       <http://purl.org/dc/terms/>
PREFIX dcat:      <http://www.w3.org/ns/dcat#>
PREFIX geosparql: <http://www.opengis.net/ont/geosparql#>
PREFIX schema:    <https://schema.org/>
PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?aspect ?value
WHERE {
  ?fdo a dcat:Dataset .
  {
    ?fdo dcat:bbox ?value . BIND("bbox" AS ?aspect)
  } UNION {
    ?fdo schema:latitude ?value . BIND("latitude" AS ?aspect)
  } UNION {
    ?fdo schema:longitude ?value . BIND("longitude" AS ?aspect)
  } UNION {
    ?fdo dct:spatial ?value . BIND("spatial (OSM/URI)" AS ?aspect)
  } UNION {
    ?fdo geosparql:hasGeometry ?geom .
    ?geom geosparql:asWKT ?value . BIND("WKT geometry" AS ?aspect)
  } UNION {
    ?fdo dct:temporal ?period .
    ?period rdfs:label ?value . BIND("temporal label" AS ?aspect)
  } UNION {
    ?fdo dct:temporal ?period .
    ?period dcat:startDate ?value . BIND("temporal start" AS ?aspect)
  } UNION {
    ?fdo dct:temporal ?period .
    ?period dcat:endDate ?value . BIND("temporal end" AS ?aspect)
  }
}
ORDER BY ?aspect
"""

display(Markdown("## 4.5 Spatial & Temporal Coverage"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_SPATIOTEMPORAL)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_SPATIOTEMPORAL)
_ = None  # suppress Out[] cell output

### 4.6 Keywords / Subjects

In [ ]:
# Keywords: FDOx uses both dcat:keyword and dct:subject (often duplicated)
# Mix of plain strings and Wikidata URIs
Q_KEYWORDS = """
PREFIX dcat:  <http://www.w3.org/ns/dcat#>
PREFIX dct:   <http://purl.org/dc/terms/>

SELECT DISTINCT ?keyword ?source
WHERE {
  ?fdo a dcat:Dataset .
  {
    ?fdo dcat:keyword ?keyword . BIND("dcat:keyword" AS ?source)
  } UNION {
    ?fdo dct:subject ?keyword . BIND("dct:subject" AS ?source)
  }
}
ORDER BY ?source ?keyword
"""

display(Markdown("## 4.6 Keywords & Subjects"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_KEYWORDS)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_KEYWORDS)
_ = None  # suppress Out[] cell output

### 4.7 Related Resources & Links

In [ ]:
# Media type distribution — useful to understand what file types are in an FDO
Q_MEDIATYPE_STATS = """
PREFIX dcat:  <http://www.w3.org/ns/dcat#>
PREFIX fdo:   <https://w3id.org/fdo-squirrel/>

SELECT ?mediaType ?role (COUNT(?dist) AS ?count)
       (SUM(?bytes) AS ?totalBytes)
       (MIN(?bytes) AS ?minBytes) (MAX(?bytes) AS ?maxBytes)
WHERE {
  ?fdo a dcat:Dataset .
  ?fdo dcat:distribution ?dist .
  OPTIONAL { ?dist dcat:mediaType ?mediaType }
  OPTIONAL { ?dist fdo:role       ?role }
  OPTIONAL { ?dist dcat:byteSize  ?bytes }
}
GROUP BY ?mediaType ?role
ORDER BY DESC(?count)
"""

display(Markdown("## 4.7 Media Type Distribution per Role"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_MEDIATYPE_STATS)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_MEDIATYPE_STATS)
_ = None  # suppress Out[] cell output

### 4.8 Graph Statistics Summary

In [ ]:
# Graph statistics: what types are in the graph and how many instances
Q_STATS = """
SELECT ?type (COUNT(?instance) AS ?count)
WHERE {
  ?instance a ?type .
  FILTER(?type != <http://www.w3.org/2002/07/owl#NamedIndividual>)
}
GROUP BY ?type
ORDER BY DESC(?count)
"""

display(Markdown("## 4.8 Graph Statistics: Instances per Type"))
display(Markdown("**Software FDO**"))
sparql_table(g_sw, Q_STATS)
display(Markdown("**3D Model FDO**"))
sparql_table(g_3d, Q_STATS)
_ = None  # suppress Out[] cell output

---
## 5 — Flexible SPARQL Sandbox

Write and run your own queries here against either graph.

In [ ]:
# ── Choose graph: g_sw (Software) or g_3d (3D Model) ────────────────────────
ACTIVE_GRAPH = g_sw

MY_QUERY = """
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dct:  <http://purl.org/dc/terms/>

SELECT ?s ?p ?o
WHERE {
  ?s ?p ?o .
}
LIMIT 20
"""

sparql_table(ACTIVE_GRAPH, MY_QUERY, label="Custom Query Results")

---
## 6 — Export Results to CSV

In [ ]:
# ── Export basic metadata for both FDOs ─────────────────────────────────────
df_sw = sparql_table(g_sw, Q_BASIC, show=False)
df_3d = sparql_table(g_3d, Q_BASIC, show=False)

df_sw.to_csv("fdox_software_metadata.csv", index=False)
df_3d.to_csv("fdox_3dmodel_metadata.csv", index=False)

print("✅ Exported: fdox_software_metadata.csv")
print("✅ Exported: fdox_3dmodel_metadata.csv")
